# Preprocessing Audio
Langkah pembersihan:
1. Format murni WAV 16kHz Mono
2. Hapus DC Offset
3. Trim Silence 
4. Pre-Emphasis
5. Normalisasi Volume

In [ ]:
import os
import glob
import librosa
import librosa.display
import soundfile as sf
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('Library berhasil dimuat!')

### Fungsi Pembersihan Suara

In [ ]:
def preprocess_audio(file_path, target_sr=16000):
    y, sr = librosa.load(file_path, sr=target_sr, mono=True)
    y = y - np.mean(y)
    y_trimmed, _ = librosa.effects.trim(y, top_db=20)
    y_preemph = librosa.effects.preemphasis(y_trimmed)
    y_clean = librosa.util.normalize(y_preemph)
    return y, y_clean, sr


### Batch Processing

In [ ]:
raw_dir = 'dataset/raw'
prep_dir = 'dataset/preprocessed'
classes = ['horas', 'sampurasun', 'adilkatalino', 'wawawa', 'kulanuwun', 'tabea', 'apahabarpian', 'noise']
total_processed = 0
for cls in classes:
    os.makedirs(os.path.join(prep_dir, cls), exist_ok=True)
    files = glob.glob(os.path.join(raw_dir, cls, '*.*'))
    print(f'Memproses {len(files)} file di kelas: {cls}...')
    for file_path in files:
        try:
            _, y_clean, sr = preprocess_audio(file_path)
            base_name = os.path.splitext(os.path.basename(file_path))[0]
            new_path = os.path.join(prep_dir, cls, f"{base_name}.wav")
            sf.write(new_path, y_clean, sr)
            total_processed += 1
        except Exception as e:
            print(f"Gagal: {file_path}")
print(f'\nSUKSES! Total {total_processed} file berhasil dicuci.')

### Tinjauan Hasil: Before vs After

In [ ]:
sample_raws = glob.glob('dataset/raw/*/*.*')
if sample_raws:
    sample_raw = sample_raws[0]
    base_name = os.path.splitext(os.path.basename(sample_raw))[0]
    cls_name = os.path.basename(os.path.dirname(sample_raw))
    sample_prep = os.path.join('dataset/preprocessed', cls_name, f"{base_name}.wav")
    
    y_original, sr_ori = librosa.load(sample_raw, sr=None)
    y_clean, sr_clean = librosa.load(sample_prep, sr=None)
    
    plt.figure(figsize=(14, 5))
    plt.subplot(1, 2, 1)
    librosa.display.waveshow(y_original, sr=sr_ori, alpha=0.5, color='gray')
    plt.title('Sebelum: Data Asli (Kotor & Ada Hening)')
    plt.subplot(1, 2, 2)
    librosa.display.waveshow(y_clean, sr=sr_clean, color='red', alpha=0.7)
    plt.title('Sesudah: Bersih (Trimmed, DC Offset, Pre-Emphasis)')
    plt.tight_layout()
    plt.show()
else:
    print('Belum ada file.')

### Tinjauan Hasil: 8 Kelas Sesudah Dibersihkan

In [ ]:
plt.figure(figsize=(16, 12))
for i, cls in enumerate(classes):
    files = glob.glob(f'dataset/preprocessed/{cls}/*.wav')
    if files:
        try:
            y_clean, sr = librosa.load(files[0], sr=16000)
            plt.subplot(4, 2, i+1)
            librosa.display.waveshow(y_clean, sr=sr, alpha=0.7, color='teal')
            plt.title(f'Waveform Bersih: {cls.upper()}')
        except:
            pass
    else:
        plt.subplot(4, 2, i+1)
        plt.title(f'Belum ada data: {cls.upper()}')
        plt.axis('off')
plt.tight_layout()
plt.show()
